# Prova de connexió a l'ERP

A la unitat 2 es treballa amb l'ERP de **Distribucions Es Pla, SL**, una base de dades PostgreSQL a Azure. Aquest notebook només
comprova que hi podeu entrar: si arribeu al final amb el missatge "Connexió correcta", ja ho teniu tot a punt per a la sessió.

Cal executar-lo a **JupyterHub de l'institut** (no a Colab): l'ERP només accepta connexions des de la xarxa de l'institut.

Executeu les cel·les per ordre (Maj + Retorn).

## 1. El vostre correu

Escriviu el correu de l'institut entre les cometes.

In [ ]:
USUARI = "nom@alumnes.politecnicllevant.cat"

## 2. Iniciar la sessió

En executar la cel·la surt un missatge amb una adreça i un codi:

1. Obriu l'adreça en una altra pestanya del navegador.
2. Escriviu-hi el codi.
3. Inicieu la sessió amb el **compte de l'institut** (no un compte personal de Microsoft) i accepteu.

La primera vegada, Microsoft demana configurar la **verificació en dos passos**. Proposa l'aplicació Microsoft Authenticator,
però també es pot fer servir Google Authenticator o una altra: triar "Vull utilitzar una altra aplicació d'autenticació"
(*I want to use a different authenticator app*) i escanejar el codi QR.

Quan acabeu, la cel·la continua sola.

In [ ]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo

import psycopg
from azure.identity import DeviceCodeCredential

HOST = "pg-5104-distribucio.postgres.database.azure.com"
TENANT = "politecnicllevant.cat"   # el directori Entra de l'institut
ESCOP = "https://ossrdbms-aad.database.windows.net/.default"

assert USUARI != "nom@alumnes.politecnicllevant.cat", "Primer cal escriure el vostre correu a la cel·la de dalt."
USUARI = USUARI.strip().lower()
TOKEN = os.environ.get("ERP_TOKEN") or DeviceCodeCredential(tenant_id=TENANT).get_token(ESCOP).token
print("Sessió iniciada: tenim el token.")

## 3. Connectar-se a l'ERP

In [ ]:
PISTES = {
    "timeout": "No respon el servidor. Comproveu que treballeu a JupyterHub de l'institut, no a Colab ni a l'ordinador de casa.",
    "resolve host": "No es troba el servidor: no toqueu el valor de HOST.",
    "password authentication failed": "El correu no coincideix amb el compte amb què heu iniciat la sessió, o no teniu accés. Reviseu USUARI.",
    "token": "El token no és vàlid o ha caducat. Torneu a executar la cel·la de l'inici de sessió.",
}
try:
    con = psycopg.connect(host=HOST, dbname="erp_distribucio", user=USUARI, password=TOKEN,
                          sslmode="require", connect_timeout=15)
except psycopg.Error as e:
    missatge = str(e)
    pista = next((p for clau, p in PISTES.items() if clau in missatge.lower()), "Copieu el missatge i porteu-lo a classe.")
    print("No s'ha pogut connectar.\n\nMissatge:", missatge, "\n\nQuè fer:", pista)
    raise SystemExit
print("Connectat com a", con.execute("SELECT current_user").fetchone()[0])

## 4. Una primera consulta

Quants clients té l'empresa de cada tipus?

In [ ]:
for tipus, n in con.execute("SELECT tipus, count(*) FROM erp.clients GROUP BY tipus ORDER BY 2 DESC"):
    print(f"{tipus:<12} {n}")

In [ ]:
con.close()
print(f"Connexió correcta. {USUARI}, {datetime.now(ZoneInfo("Europe/Madrid")):%d/%m/%Y %H:%M}")